In [ ]:
# Ensure helper .py files locations:
# - cartpole_wind_pch.py in project root.
# - ucbvi.py, ucbq.py in <project_root>/causalgym/causal_gym/algorithms/
# - train_cartpole_ucbvi.py (for Discretiser) in this directory.

# %pip install -q gymnasium matplotlib numpy tqdm # tqdm is optional

import numpy as np
import matplotlib.pyplot as plt
import os
import sys

# Path to the directory containing this notebook (causal2/causalgym/test/learning/)
# os.getcwd() should typically be the notebook's directory when run interactively.
current_notebook_actual_dir = os.getcwd()

# Path to the project root (three levels up from causalgym/test/learning/)
project_root_from_nb = os.path.abspath(os.path.join(current_notebook_actual_dir, "..", "..", ".."))
if project_root_from_nb not in sys.path:
    sys.path.insert(0, project_root_from_nb) 
    print(f"Added project root to sys.path (for cartpole_wind_pch.py): {project_root_from_nb}")

# Path to the algorithms directory for UCBVI, UCBQ
# <project_root>/causalgym/causal_gym/algorithms/
algorithms_dir_from_nb = os.path.join(project_root_from_nb, "causalgym", "causal_gym", "algorithms")
if algorithms_dir_from_nb not in sys.path:
    sys.path.insert(0, algorithms_dir_from_nb)
    print(f"Added algorithms directory to sys.path (for ucbvi.py, ucbq.py): {algorithms_dir_from_nb}")

# Discretiser is in train_cartpole_ucbvi.py, which is now in the same directory as this notebook.
# Adding the current directory explicitly to sys.path is usually not needed if running from here,
# but can be done for robustness if scripts are called from other locations.
if current_notebook_actual_dir not in sys.path:
    sys.path.insert(0, current_notebook_actual_dir)
    print(f"Added notebook directory to sys.path (for Discretiser): {current_notebook_actual_dir}")

try:
    from train_cartpole_ucbvi import Discretiser # From this directory
    from cartpole_wind_pch import CartPoleWindPCH # From project_root
    from ucbvi import UCBVI                     # From algorithms_dir
    # from ucbq import UCBQ 
    print("Imports successful.")
except ImportError as e:
    print(f"ImportError: {e}. Please check paths and file locations.")
    print(f"Attempted project root: {project_root_from_nb}")
    print(f"Attempted algorithms_dir: {algorithms_dir_from_nb}")
    print(f"Attempted notebook dir for Discretiser: {current_notebook_actual_dir}")
    print(f"Current sys.path: {sys.path}")

# Optional: for progress bar
try:
    from tqdm.auto import tqdm
except ImportError:
    print("tqdm not found, will run without progress bars. To install: %pip install tqdm")
    def tqdm(iterable, **kwargs):
        return iterable

print("Setup Complete.")

In [ ]:
# Configuration for a quick demo run
N_EPISODES = 200
HORIZON = 200
BINS_PER_DIM = 8 # As per prompt for demo
DELTA = 0.1
EPSILON = 0.1 # UCBVI uses this
SEED = 42

# 1. Instantiate Environment
env = CartPoleWindPCH()

# 2. Instantiate Discretiser
discretiser = Discretiser(bins_per_dim=BINS_PER_DIM)
print(f"Discretiser created with {discretiser.n_states} states.")

# 3. Instantiate Agent (UCBVI for this demo, as per prompt)
agent = UCBVI(
    num_states=discretiser.n_states,
    n_actions=env.action_space.n, # Should be 2
    horizon=HORIZON,
    delta=DELTA,
    epsilon=EPSILON,
    seed=SEED
)
print(f"UCBVI agent instantiated.")

print("Environment and Agent Instantiation Complete.")

In [ ]:
episode_returns_notebook = []
episode_regrets_notebook = [] # Regret = sum of (1 - reward_step) for each episode

print(f"Starting training for {N_EPISODES} episodes...")

for ep in tqdm(range(N_EPISODES)):
    obs_continuous, _ = env.reset(seed=SEED + ep)
    s = discretiser(obs_continuous)
    ep_return = 0.0
    ep_regret_steps = 0.0

    agent.plan() # UCBVI plans at the start of each episode

    for t in range(HORIZON):
        x_intended = np.argmax(agent.V[s, :])
        a_to_apply = agent.act(s, x_intended)
        
        obs2_continuous, r, terminated, truncated, info = env.do(a_to_apply)
        s2 = discretiser(obs2_continuous)
        done = terminated or truncated
        
        realised_action = info.get("realised_action", a_to_apply)
        agent.update(s, x_intended, realised_action, r, s2)
        
        ep_return += r
        ep_regret_steps += (1.0 - r) # Max reward is 1.0 per step for CartPole
        s = s2
        
        if done:
            break
            
    episode_returns_notebook.append(ep_return)
    episode_regrets_notebook.append(ep_regret_steps)

env.close()
print("Training Complete.")
print(f"Sample returns (first 5): {episode_returns_notebook[:5]}")
print(f"Sample regrets (first 5): {episode_regrets_notebook[:5]}")

In [ ]:
# Calculate moving average for returns for smoother plot
window_size = 20
if len(episode_returns_notebook) >= window_size:
    moving_avg_returns = np.convolve(episode_returns_notebook, np.ones(window_size)/window_size, mode='valid')
else:
    moving_avg_returns = episode_returns_notebook # Not enough data for moving average

# Calculate cumulative regret
cumulative_regrets_notebook = np.cumsum(episode_regrets_notebook)

# Plotting
fig, axs = plt.subplots(2, 1, figsize=(10, 8))

# Plot 1: Average Episode Return
axs[0].plot(moving_avg_returns, label=f'Moving Avg (Win={window_size})' if len(episode_returns_notebook) >= window_size else 'Episode Return')
axs[0].plot(episode_returns_notebook, alpha=0.3, label='Raw Episode Return')
axs[0].set_title(f'Episode Returns (UCBVI on CartPoleWind, Bins={BINS_PER_DIM})')
axs[0].set_xlabel('Episode')
axs[0].set_ylabel('Return')
axs[0].grid(True)
axs[0].legend()

# Plot 2: Cumulative Regret
axs[1].plot(cumulative_regrets_notebook)
axs[1].set_title('Cumulative Regret vs. Episodes')
axs[1].set_xlabel('Episode')
axs[1].set_ylabel('Cumulative Regret (Sum of 1-r)')
axs[1].grid(True)

plt.tight_layout()
plt.show()

print("Plotting Complete.")

# Optional: Animation of one trained rollout (only if running locally)
# This part might be platform-dependent for display and requires render_mode='human' or similar during env creation.
# For simplicity and Colab compatibility, we'll skip direct animation playback here.
# To generate a GIF, one would typically re-run a rollout with render_mode='rgb_array' 
# and use matplotlib.animation.FuncAnimation, similar to train_cartpole_ucbvi.py.

# print("\nTo generate an animation/GIF, you can adapt the GIF generation code from train_cartpole_ucbvi.py")
# print("Example (conceptual - requires agent to be trained):")
# print("frames = []")
# print("gif_env = CartPoleWindPCH(render_mode='rgb_array')")
# print("obs_gif, _ = gif_env.reset(seed=SEED + N_EPISODES + 1)")
# print("s_gif = discretiser(obs_gif)")
# print("agent.plan() # Re-plan with final Q/V values")
# print("for _ in range(HORIZON):")
# print("    frame = gif_env.render()")
# print("    if frame is None: break")
# print("    frames.append(frame)")
# print("    x_intended_gif = np.argmax(agent.V[s_gif, :])")
# print("    a_gif = agent.act(s_gif, x_intended_gif)")
# print("    obs_next_gif, _, term, trunc, _ = gif_env.do(a_gif)")
# print("    s_gif = discretiser(obs_next_gif)")
# print("    if term or trunc: break")
# print("gif_env.close()")
# print("# ... then use matplotlib.animation to save 'frames' list ...")
